# NB_bishop_ch19_figures
## Key Figures for Bishop Chapter 19 -- Autoencoders and VAEs

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/danpele/NEURAL_BIZ/blob/master/Bishop_Ch_19/NB_bishop_ch19_figures.ipynb)

Generate publication-quality figures for autoencoder architecture, VAE latent space, reparameterization trick, and reconstruction grids.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib
from matplotlib.patches import FancyArrowPatch

matplotlib.rcParams['figure.facecolor'] = 'none'
matplotlib.rcParams['axes.facecolor'] = 'none'
matplotlib.rcParams['savefig.facecolor'] = 'none'
matplotlib.rcParams['savefig.transparent'] = True
matplotlib.rcParams['axes.grid'] = False
matplotlib.rcParams['legend.loc'] = 'upper center'
matplotlib.rcParams['legend.framealpha'] = 0.0

COLORS = {'blue': '#1A3A6E', 'red': '#CD0000', 'green': '#2E7D32', 'amber': '#B5853F'}

def save_fig(fig, name, chart_dir='../../charts'):
    for ax in fig.get_axes():
        ax.grid(False)
        legend = ax.get_legend()
        if legend:
            legend.set_bbox_to_anchor((0.5, -0.15))
            legend.set_loc('upper center')
            legend._ncols = min(len(legend.get_texts()), 4)
    fig.savefig(f'{chart_dir}/{name}.pdf', bbox_inches='tight', transparent=True)
    fig.savefig(f'{chart_dir}/{name}.png', bbox_inches='tight', transparent=True, dpi=150)
    print(f'Saved {chart_dir}/{name}.pdf and .png')

np.random.seed(42)

## Figure 19.1 -- Autoencoder Architecture

Input -> Encoder (narrowing layers) -> Bottleneck z -> Decoder (widening layers) -> Output.

In [ ]:
fig, ax = plt.subplots(figsize=(14, 6))
ax.set_xlim(-1, 15)
ax.set_ylim(-1, 7)
ax.axis('off')

# Layer widths (visual height of each layer block)
encoder_widths = [5.0, 3.5, 2.0]  # narrowing
bottleneck_width = 1.0
decoder_widths = [2.0, 3.5, 5.0]  # widening

layer_x_positions = [0, 2, 4, 6.5, 9, 11, 13]
all_widths = encoder_widths + [bottleneck_width] + decoder_widths
layer_labels = ['Input\n$\mathbf{x}$', '256', '64', '$\mathbf{z}$\n(latent)', '64', '256',
                'Output\n$\hat{\mathbf{x}}$']
layer_colors = [COLORS['blue'], COLORS['blue'], COLORS['blue'],
                COLORS['amber'],
                COLORS['red'], COLORS['red'], COLORS['red']]
layer_fc = ['#D6E4F0', '#D6E4F0', '#D6E4F0',
            '#FFF3E0',
            '#FCE4EC', '#FCE4EC', '#FCE4EC']

center_y = 3.0
box_w = 1.2

for i, (x, w, label, ec, fc) in enumerate(zip(layer_x_positions, all_widths,
                                                layer_labels, layer_colors, layer_fc)):
    h = w
    y = center_y - h / 2
    box = mpatches.FancyBboxPatch((x, y), box_w, h, boxstyle='round,pad=0.1',
                                   facecolor=fc, edgecolor=ec, linewidth=2, alpha=0.85)
    ax.add_patch(box)
    ax.text(x + box_w / 2, center_y, label, ha='center', va='center',
            fontsize=9, fontweight='bold', color=ec)

    # Draw arrow to next layer
    if i < len(layer_x_positions) - 1:
        next_x = layer_x_positions[i + 1]
        ax.annotate('', xy=(next_x, center_y), xytext=(x + box_w, center_y),
                    arrowprops=dict(arrowstyle='->', color='#555555', lw=1.8))

# Labels for encoder / decoder
ax.text(3.0, 6.2, 'Encoder $q_\phi(\mathbf{z}|\mathbf{x})$', ha='center',
        fontsize=13, fontweight='bold', color=COLORS['blue'])
ax.annotate('', xy=(0.5, 5.9), xytext=(5.5, 5.9),
            arrowprops=dict(arrowstyle='<->', color=COLORS['blue'], lw=1.5))

ax.text(11.5, 6.2, 'Decoder $p_\\theta(\mathbf{x}|\mathbf{z})$', ha='center',
        fontsize=13, fontweight='bold', color=COLORS['red'])
ax.annotate('', xy=(8.5, 5.9), xytext=(13.7, 5.9),
            arrowprops=dict(arrowstyle='<->', color=COLORS['red'], lw=1.5))

ax.set_title('Autoencoder Architecture', fontweight='bold', fontsize=15, pad=15)

fig.tight_layout()
save_fig(fig, 'fig19_1_autoencoder')
plt.show()

## Figure 19 -- VAE Latent Space

2D latent space with colored clusters (simulating digit-like class separation).

In [ ]:
np.random.seed(42)
n_classes = 10
n_per_class = 80

# Generate cluster centers in a circle
angles = np.linspace(0, 2 * np.pi, n_classes, endpoint=False)
radius = 3.0
centers = np.column_stack([radius * np.cos(angles), radius * np.sin(angles)])

# Generate points around each center
all_points = []
all_labels = []
for i in range(n_classes):
    pts = centers[i] + 0.5 * np.random.randn(n_per_class, 2)
    all_points.append(pts)
    all_labels.extend([i] * n_per_class)

all_points = np.vstack(all_points)
all_labels = np.array(all_labels)

# Color map
cmap = plt.cm.tab10

fig, ax = plt.subplots(figsize=(8, 8))
for i in range(n_classes):
    mask = all_labels == i
    ax.scatter(all_points[mask, 0], all_points[mask, 1],
               c=[cmap(i)], s=12, alpha=0.6, label=f'Class {i}')
    ax.text(centers[i, 0], centers[i, 1], str(i), fontsize=14, fontweight='bold',
            ha='center', va='center', color=cmap(i),
            bbox=dict(boxstyle='round,pad=0.2', facecolor='white', edgecolor=cmap(i), alpha=0.8))

ax.set_xlabel('$z_1$', fontsize=13)
ax.set_ylabel('$z_2$', fontsize=13)
ax.set_title('VAE Latent Space (2D)', fontweight='bold', fontsize=14)
ax.set_aspect('equal')
ax.legend(fontsize=8, ncol=5)

fig.tight_layout()
save_fig(fig, 'fig19_vae_latent')
plt.show()

## Figure 19 -- Reparameterization Trick

Diagram: encoder outputs $\mu$, $\sigma$ -> sample $\varepsilon \sim \mathcal{N}(0,1)$ -> compute $z = \mu + \sigma \varepsilon$.

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))
ax.set_xlim(-1, 15)
ax.set_ylim(-0.5, 7)
ax.axis('off')

def draw_rbox(ax, x, y, w, h, label, fc, ec, fs=11):
    box = mpatches.FancyBboxPatch((x, y), w, h, boxstyle='round,pad=0.12',
                                   facecolor=fc, edgecolor=ec, linewidth=2)
    ax.add_patch(box)
    ax.text(x + w/2, y + h/2, label, ha='center', va='center',
            fontsize=fs, fontweight='bold', color=ec)

# Encoder output
draw_rbox(ax, 0, 2.5, 2.5, 2, 'Encoder\n$q_\phi(z|x)$', '#D6E4F0', COLORS['blue'], 12)

# Mu
draw_rbox(ax, 4, 4.2, 1.5, 1, '$\mu$', '#E8F5E9', COLORS['green'], 14)

# Sigma
draw_rbox(ax, 4, 1.8, 1.5, 1, '$\sigma$', '#FFF3E0', COLORS['amber'], 14)

# Arrows from encoder to mu, sigma
ax.annotate('', xy=(4, 4.7), xytext=(2.5, 4.0),
            arrowprops=dict(arrowstyle='->', color=COLORS['blue'], lw=2))
ax.annotate('', xy=(4, 2.3), xytext=(2.5, 3.0),
            arrowprops=dict(arrowstyle='->', color=COLORS['blue'], lw=2))

# Epsilon (sampled noise)
draw_rbox(ax, 4, -0.2, 1.5, 1, '$\\varepsilon \sim \mathcal{N}(0,1)$', '#FCE4EC', COLORS['red'], 10)

# Multiplication: sigma * epsilon
circle_mult = plt.Circle((7.5, 1.5), 0.35, facecolor='white', edgecolor='black', linewidth=2, zorder=5)
ax.add_patch(circle_mult)
ax.text(7.5, 1.5, '$\\times$', ha='center', va='center', fontsize=16, fontweight='bold', zorder=6)

# Arrows to multiplication
ax.annotate('', xy=(7.15, 1.8), xytext=(5.5, 2.3),
            arrowprops=dict(arrowstyle='->', color=COLORS['amber'], lw=2))
ax.annotate('', xy=(7.3, 1.2), xytext=(5.5, 0.3),
            arrowprops=dict(arrowstyle='->', color=COLORS['red'], lw=2))

# Addition: mu + sigma*epsilon
circle_add = plt.Circle((9.5, 3.5), 0.35, facecolor='white', edgecolor='black', linewidth=2, zorder=5)
ax.add_patch(circle_add)
ax.text(9.5, 3.5, '$+$', ha='center', va='center', fontsize=16, fontweight='bold', zorder=6)

# Arrow from mu to addition
ax.annotate('', xy=(9.2, 3.8), xytext=(5.5, 4.7),
            arrowprops=dict(arrowstyle='->', color=COLORS['green'], lw=2))

# Arrow from multiplication to addition
ax.annotate('', xy=(9.3, 3.2), xytext=(7.85, 1.7),
            arrowprops=dict(arrowstyle='->', color='black', lw=2))

# z output
draw_rbox(ax, 11, 2.8, 2.5, 1.5, '$\mathbf{z} = \mu + \sigma\\varepsilon$',
          '#E8E8E8', '#333333', 13)

# Arrow from addition to z
ax.annotate('', xy=(11, 3.5), xytext=(9.85, 3.5),
            arrowprops=dict(arrowstyle='->', color='black', lw=2))

# Gradient flow annotation
ax.annotate('Gradients flow through\n(deterministic path)',
            xy=(9.5, 5.5), fontsize=10, ha='center', color='#555555', fontstyle='italic',
            bbox=dict(boxstyle='round,pad=0.3', facecolor='#FFFDE7', edgecolor='#888888', alpha=0.8))

ax.set_title('VAE Reparameterization Trick', fontweight='bold', fontsize=15, pad=15)

fig.tight_layout()
save_fig(fig, 'fig19_vae_reparameterization')
plt.show()

## Figure 19 -- Reconstruction Grid

Top row: original patterns, bottom row: reconstructions (using random noise patterns as placeholders).

In [ ]:
np.random.seed(42)
n_samples = 8
img_size = 16

# Create simple structured patterns as "originals"
originals = []
for i in range(n_samples):
    img = np.zeros((img_size, img_size))
    # Different geometric patterns
    if i % 4 == 0:  # horizontal bar
        r = np.random.randint(3, img_size - 3)
        img[r-1:r+2, 2:-2] = 1.0
    elif i % 4 == 1:  # vertical bar
        c = np.random.randint(3, img_size - 3)
        img[2:-2, c-1:c+2] = 1.0
    elif i % 4 == 2:  # diagonal
        for k in range(img_size):
            if 0 <= k < img_size:
                img[k, min(k, img_size-1)] = 1.0
                if k+1 < img_size:
                    img[k, min(k+1, img_size-1)] = 0.5
    else:  # cross
        img[img_size//2-1:img_size//2+2, 2:-2] = 1.0
        img[2:-2, img_size//2-1:img_size//2+2] = 1.0
    originals.append(img)

# Simulate reconstructions (originals + noise, slightly blurred)
reconstructions = []
for img in originals:
    noisy = img + 0.15 * np.random.randn(img_size, img_size)
    # Simple blur via averaging with neighbors
    blurred = np.copy(noisy)
    blurred[1:-1, 1:-1] = 0.5 * noisy[1:-1, 1:-1] + \
                          0.125 * (noisy[:-2, 1:-1] + noisy[2:, 1:-1] +
                                   noisy[1:-1, :-2] + noisy[1:-1, 2:])
    reconstructions.append(np.clip(blurred, 0, 1))

fig, axes = plt.subplots(2, n_samples, figsize=(16, 4.5))

for j in range(n_samples):
    axes[0, j].imshow(originals[j], cmap='Blues', vmin=0, vmax=1, interpolation='nearest')
    axes[0, j].axis('off')
    if j == 0:
        axes[0, j].set_ylabel('Original', fontsize=12, fontweight='bold', color=COLORS['blue'])

    axes[1, j].imshow(reconstructions[j], cmap='Reds', vmin=0, vmax=1, interpolation='nearest')
    axes[1, j].axis('off')
    if j == 0:
        axes[1, j].set_ylabel('Reconstructed', fontsize=12, fontweight='bold', color=COLORS['red'])

# Row labels using text on the left side
fig.text(0.02, 0.72, 'Original', fontsize=13, fontweight='bold', color=COLORS['blue'],
         ha='center', va='center', rotation=90)
fig.text(0.02, 0.28, 'Reconstructed', fontsize=13, fontweight='bold', color=COLORS['red'],
         ha='center', va='center', rotation=90)

fig.suptitle('Autoencoder: Input vs Reconstruction', fontweight='bold', fontsize=14, y=1.02)
fig.tight_layout()
save_fig(fig, 'fig19_reconstruction')
plt.show()

## Summary

- **fig19_1**: Autoencoder architecture showing encoder-bottleneck-decoder structure
- **fig19_vae_latent**: 2D latent space visualization with class clusters
- **fig19_vae_reparameterization**: Reparameterization trick enabling gradient flow
- **fig19_reconstruction**: Original vs reconstructed pattern comparison grid